## Setup

In [ ]:
import os
from dotenv import load_dotenv
import numpy as np
import pandas as pd
from datetime import datetime
from astropy.time import Time

pd.set_option('mode.copy_on_write', True)
pd.set_option('future.no_silent_downcasting', True)

In [ ]:
load_dotenv()

RAW_DATA_PATH = os.getenv('MAG_RAW_PATH')
TREATED_GLOBAL_PATH = os.getenv('MAG_TREATED_GLOBAL_PATH')
TREATED_BY_REGION_PATH = os.getenv('MAG_TREATED_BY_REGION_PATH')

BEGIN_DATE = "20100501_000000"
END_DATE = "20240921_235900"

# Lista completa de features do JSOC extraídas no scrape.
SHARP_PARAMS = [
    'USFLUX', 'MEANJZH', 'TOTUSJH', 'ABSNJZH', 'SAVNCPP', 'MEANPOT',
    'TOTPOT', 'MEANALP', 'SHRGT45', 'TOTUSJZ', 'MEANSHR', 'MEANJZD',
    'MEANGAM', 'MEANGBT', 'MEANGBZ', 'MEANGBH', 'TOTFZ', 'TOTBSQ',
    'TOTSYQ', 'R_VALUE', 'AREA_ACR'
]

## Reading Data

In [ ]:
def parse_tai_to_utc(tai_series: pd.Series) -> pd.Series:
    iso_strings = tai_series.str.replace('.', '-', regex=False).str.replace('_', 'T', regex=False)
    t_tai = Time(iso_strings.tolist(), format='isot', scale='tai')
    return pd.to_datetime(t_tai.utc.isot)

In [ ]:
date_format = "%Y%m%d_%H%M%S"
start_dt = datetime.strptime(BEGIN_DATE, date_format)
end_dt = datetime.strptime(END_DATE, date_format)

df_raw_list = []
current_start = start_dt

while current_start < end_dt:
    current_end = current_start + pd.DateOffset(months=1)
    if current_end > end_dt:
        current_end = end_dt

    start_year = str(current_start.year)
    year_dir = os.path.join(RAW_DATA_PATH, start_year)

    start_str = current_start.strftime("%Y%m%d_%H%M%S")
    end_str = current_end.strftime("%Y%m%d_%H%M%S")

    file_name = f"jsoc_data_{start_str}_TAI_to_{end_str}_TAI.csv"
    full_path = os.path.join(year_dir, file_name)

    if os.path.exists(full_path):
        df = pd.read_csv(full_path)

        df.loc[:, 'REGION_ID'] = df['DATASET_QUERY'].str.extract(r'\[(\d+)\]')

        df.loc[:, 'T_REC'] = df['T_REC'].str.replace('_TAI', '', regex=False)
        df.loc[:, 'T_REC'] = parse_tai_to_utc(df['T_REC'])
        df = df.rename(columns={'T_REC': 'ds'})

        df_raw_list.append(df)

    current_start = current_end

df_mag_raw = pd.concat(df_raw_list, ignore_index=True)

In [ ]:
df_mag_raw

## Treating Data

### 1. Tratamento de Qualidade e Dados Ausentes

Limpeza inicial baseada nos defeitos relatados na literatura para os dados espaciais (SHARPs). A base é filtrada para descartar medições comprometidas por anomalias do sensor, valores infinitos e dados ausentes nas features centrais.

In [ ]:
df_mag_clean = df_mag_raw.copy()
initial_len = len(df_mag_clean)

In [ ]:
# =============================================================================
# 1.1 FILTRO DE QUALIDADE (QUALITY FLAG)
# Converte a flag hexadecimal para inteiro e descarta tudo que for diferente de 0
# =============================================================================
df_mag_clean.loc[:, 'QUALITY'] = df_mag_clean['QUALITY'].apply(
    lambda x: int(str(x), 16) if isinstance(x, str) and str(x).startswith('0x') else pd.to_numeric(x, errors='coerce')
)
mask_quality = df_mag_clean['QUALITY'] == 0
len_bad_quality = initial_len - mask_quality.sum()
df_mag_clean = df_mag_clean[mask_quality].copy()

In [ ]:
# =============================================================================
# 1.2 CONVERSÃO NUMÉRICA E TRATAMENTO DE DADOS AUSENTES/INFINITOS
# Força a conversão das strings para números e deleta linhas que não possuem as features essenciais
# =============================================================================
cols_to_numeric = SHARP_PARAMS + ['LON_FWT', 'LAT_FWT', 'LON_MIN', 'LON_MAX']

for col in cols_to_numeric:
    if col in df_mag_clean.columns:
        df_mag_clean.loc[:, col] = pd.to_numeric(df_mag_clean[col], errors='coerce')

# Substituir valores infinitos por NaN
df_mag_clean = df_mag_clean.replace([np.inf, -np.inf], np.nan).infer_objects(copy=False)

# Dropar apenas se as features essenciais estiverem ausentes. Variáveis como TOTFZ, TOTBSQ e TOTSYQ possuem 'InvalidKeyname' em anos mais antigos e devem ser salvas.
core_cols_to_drop = [
    'USFLUX', 'R_VALUE', 'TOTUSJH', 'TOTUSJZ', 
    'ABSNJZH', 'SHRGT45', 'MEANPOT', 'TOTPOT', 
    'MEANALP', 'LON_FWT', 'LAT_FWT'
]

cols_subset = [col for col in core_cols_to_drop if col in df_mag_clean.columns]
len_before_dropna = len(df_mag_clean)
df_mag_clean = df_mag_clean.dropna(subset=cols_subset).copy()

len_missing_incomplete = len_before_dropna - len(df_mag_clean)

print("--- Resumo do Passo 1: Filtro de Qualidade ---")
print(f"Total de registros originais: {initial_len}")
print(f"Descartados por baixa qualidade (QUALITY != 0): {len_bad_quality}")
print(f"Descartados por ausência de Features Essenciais: {len_missing_incomplete}")
print(f"Dimensão da base limpa: {df_mag_clean.shape}")

### 2. Filtro de Efeito de Projeção (Centro de Massa)

Para evitar distorções de perspectiva nas bordas do disco solar, mantemos apenas os registros onde o centro ponderado do fluxo magnético (`LON_FWT`) está entre -70° e +70° em relação ao meridiano central.

In [ ]:
# =============================================================================
# 2. FILTRO DE EFEITO DE PROJEÇÃO (LONGITUDE FWT)
# Mantém apenas os registros onde o centro de massa magnético (LON_FWT) está no intervalo de +-70 graus.
# =============================================================================
len_before_proj = len(df_mag_clean)

# Filtro focado estritamente no centro de massa (LON_FWT)
mask_projection = (df_mag_clean['LON_FWT'] >= -70) & (df_mag_clean['LON_FWT'] <= 70)
df_mag_clean = df_mag_clean[mask_projection].copy()

len_discarded_proj = len_before_proj - len(df_mag_clean)

print("--- Resumo do Passo 2: Filtro de Projeção ---")
print(f"Registros avaliados: {len_before_proj}")
print(f"Descartados por efeito de projeção (LON_FWT fora de ±70°): {len_discarded_proj}")
print(f"Dimensão da base após filtro: {df_mag_clean.shape}")

# Descartando os limitadores de borda, pois não tem mais utilidade
df_mag_clean = df_mag_clean.drop(columns=['LON_MIN', 'LON_MAX'], errors='ignore')

### 3. Auditoria de Gaps e Continuidade Temporal (Runs)

Agrupamos os dados por `REGION_ID` para calcular a diferença de tempo ($\Delta t$) entre medições consecutivas. A cadência nominal do instrumento é de 12 minutos. Gaps superiores a 36 minutos são marcados como quebras intransponíveis. Esses buracos forçam a divisão do histórico da Região Ativa em blocos independentes (`run_id`), garantindo que os modelos de janelas não aprendam saltos temporais inexistentes.

In [ ]:
# =============================================================================
# 3. AUDITORIA DE GAPS E CONTINUIDADE TEMPORAL (RUNS)
# varre a base e calcula a diferença de tempo (delta_t_min) entre cada entrada. Se essa diferença for maior que a tolerância máxima a sequência é finzalizada naquele instante. run_id é o marcador de períodos
# =============================================================================
# Ordenação cronológica estrita
df_mag_clean = df_mag_clean.sort_values(by=['REGION_ID', 'ds']).reset_index(drop=True).copy()

delta_t_min = df_mag_clean.groupby('REGION_ID')['ds'].diff().dt.total_seconds() / 60.0
df_mag_clean.loc[:, 'delta_t_min'] = delta_t_min

MAX_TOLERATED_GAP_MIN = 36.0

mask_new_run = df_mag_clean['delta_t_min'].isna() | (df_mag_clean['delta_t_min'] > MAX_TOLERATED_GAP_MIN)
df_mag_clean.loc[:, 'run_id'] = mask_new_run.cumsum()

total_regions = df_mag_clean['REGION_ID'].nunique()
total_runs = df_mag_clean['run_id'].nunique()
major_gaps_count = (df_mag_clean['delta_t_min'] > MAX_TOLERATED_GAP_MIN).sum()

print("--- Resumo do Passo 3: Gaps e Continuidade Temporal ---")
print(f"Total de Regiões Ativas únicas na base: {total_regions}")
print(f"Total de quebras de telemetria identificadas (Gaps > {MAX_TOLERATED_GAP_MIN} min): {major_gaps_count}")
print(f"Total de 'Runs' geradas (blocos temporais contínuos isolados): {total_runs}")
print(f"Aumento da cardinalidade de treino (Runs > Regiões): +{total_runs - total_regions}")

### 4. Reamostragem (Grid Temporal) e Interpolação

Se o gap for pequeno (<= a tolerancia máxima do passo 3), interpolação linear é realizada para preenchê-lo

In [ ]:
# =============================================================================
# 4. CRIAÇÃO DO GRID REGULAR E INTERPOLAÇÃO DE PEQUENOS GAPS
# =============================================================================
df_grid = df_mag_clean.copy()

df_grid.loc[:, 'ds'] = df_grid['ds'].dt.round('12min')
df_grid = df_grid.drop_duplicates(subset=['run_id', 'ds'], keep='last')
df_grid = df_grid.set_index('ds').groupby('run_id').resample('12min').asfreq()
df_grid = df_grid.drop(columns=['run_id'], errors='ignore').reset_index()

cols_to_interpolate = SHARP_PARAMS + ['LON_FWT', 'LAT_FWT']
for col in cols_to_interpolate:
    if col in df_grid.columns:
        df_grid.loc[:, col] = df_grid.groupby('run_id')[col].transform(
            lambda x: x.interpolate(method='linear', limit=2)
        )

df_grid.loc[:, 'REGION_ID'] = df_grid.groupby('run_id')['REGION_ID'].ffill()

# Restringir o drop final ao subset vital para não perder variáveis com InvalidKeynames hist
final_subset = [col for col in core_cols_to_drop if col in df_grid.columns]
len_before_drop = len(df_grid)
df_grid = df_grid.dropna(subset=final_subset).copy()

df_grid.loc[:, 'REGION_ID'] = df_grid['REGION_ID'].astype('Int64')

print("--- Resumo do Passo 4: Grid Temporal e Interpolação ---")
print(f"Linhas geradas pelo grid estrito de 12 min: {len_before_drop}")
print(f"Linhas descartadas na limpeza residual de NaNs essenciais: {len_before_drop - len(df_grid)}")
print(f"Dimensão da base final (Trusted): {df_grid.shape}")

## 5. Exporting Data

A base limpa é ramificada em duas vias arquiteturais em formato `Parquet`:
1. **Mag Regional:** O estado granular de cada região.
2. **Mag Global:** Snapshot unificado do disco solar (`_MAX` e `_SUM`) agregando todas as 21 features.

In [ ]:
# REGIONAL
df_regional = df_grid.rename(columns={'ds': 'T_REC_round'})
df_regional = df_regional.sort_values(by=['REGION_ID', 'run_id', 'T_REC_round']).reset_index(drop=True)

os.makedirs(TREATED_BY_REGION_PATH, exist_ok=True)
regional_path = os.path.join(TREATED_BY_REGION_PATH, 'treated_mag_regional.parquet')
print("Salvando Parquet Regional...")
df_regional.to_parquet(regional_path, index=False)

print(f"-> Regional Exportado para: {regional_path} | Dimensões: {df_regional.shape}")

In [ ]:
# GLOBAL (AGREGAÇÃO MAX E SUM)
cols_to_agg = [col for col in SHARP_PARAMS if col in df_regional.columns]
df_global_max = df_regional.groupby('T_REC_round')[cols_to_agg].max().add_suffix('_MAX')
# min_count=1 evita que agregações de colunas inteiramente compostas por NaNs (ex: InvalidKeyname) resultem no valor numérico 0.0, corrompendo a física do dado.
df_global_sum = df_regional.groupby('T_REC_round')[cols_to_agg].sum(min_count=1).add_suffix('_SUM')
df_global = pd.concat([df_global_max, df_global_sum], axis=1).reset_index()

os.makedirs(TREATED_GLOBAL_PATH, exist_ok=True)
global_path = os.path.join(TREATED_GLOBAL_PATH, 'treated_mag_global.parquet')
print("Salvando Parquet Global...")
df_global.to_parquet(global_path, index=False)

print(f"-> Global Exportado para:   {global_path} | Dimensões: {df_global.shape}")